In [0]:
import dlt
from pyspark.sql.functions import col

@dlt.table(
    name="gold_lap_level_race_telemetry",
    comment="Combined Gold Layer Telemetry Dataset",
    table_properties={"quality": "gold"}
)
def gold_lap_level_race_telemetry():
    # Stream NAHI, BATCH READ (dlt.read) use karein:
    results = dlt.read("silver_results").alias("result")
    drivers = dlt.read("silver_drivers").alias("driver")
    races = dlt.read("silver_races").alias("race")
    constructors = dlt.read("silver_constructors").alias("constructor")
    laps = dlt.read("silver_lap_times").alias("laps")
    circuits = dlt.read("silver_circuits").alias("circuit")

    return (
        results
        .join(drivers, "driver_id", "left")
        .join(races, "race_id", "left")
        .join(constructors, "constructor_id", "left")
        .join(
            laps, 
            (col("result.race_id") == col("laps.race_id")) & (col("result.driver_id") == col("laps.driver_id")), 
            "left"
        )
        .join(circuits, col("race.circuit_id") == col("circuit.circuit_id"), "left")
        .select(
            col("result.driver_number").alias("driver_number"),
            col("driver.driver_name").alias("name"),
            col("driver.nationality").alias("nationality"),
            col("constructor.team_name").alias("brand_name"),
            col("circuit.location").alias("location"),
            col("circuit.country").alias("country"),
            col("laps.lap_number").alias("lap_number"),
            col("laps.position").alias("lap_position"),
            col("result.laps_completed").alias("laps"),
            col("race.race_year").alias("race_year"),
            col("race.race_round").alias("race_round"),
            col("result.final_position").alias("final_position")
        )
        .orderBy(col("final_position").asc_nulls_last())
    )

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6940854969347018>, line 1
----> 1 import dlt

ModuleNotFoundError: No module named 'dlt'